# Chapter 3: Geospatial

## Ingest Notebook

This notebook fetches real-time air quality and weather data from the IQAir API for cities across the Pyrenees corridor -- the French regions of Occitanie and Nouvelle-Aquitaine, and the Spanish regions of Aragon, Catalonia and Navarre -- and loads the results into Neo4j.

City coordinates are stored as native Neo4j `point` type, enabling spatial queries with `point.distance()` directly in Cypher. The `NEIGHBORS` relationship between cities is computed in Cypher rather than Python.

The graph model is:

```
(:City {name, country, state, location})
  -[:HAS_READING]->
(:Reading {timestamp, aqi_us, aqi_category, main_pollutant,
           temperature, humidity, wind_speed})

(:City)-[:NEIGHBORS {distance_km}]-(:City)
```

City nodes are stable across runs. Each run appends new `Reading` nodes, so the database accumulates a time series. Run this notebook daily to build up readings over time.

## Prerequisites

Set the following environment variables before launching Jupyter:

```shell
export NEO4J_URI="neo4j+s://xxxxxxxx.databases.neo4j.io"
export NEO4J_USERNAME="xxxxxxxx"
export NEO4J_PASSWORD="your-password"
export NEO4J_DATABASE="xxxxxxxx"
export IQAIR_API_KEY="your-iqair-api-key"
```

Obtain a [free IQAir API key](https://www.iqair.com/commercial-air-quality-monitors/api). The free tier allows 5 requests per minute -- the notebook spaces calls 12 seconds apart.

## 1. Install Dependencies

In [1]:
%pip install ipywidgets==8.1.5 \
             jupyterlab-widgets==3.0.13 \
             neo4j==5.28.1 \
             requests==2.34.2 \
             tabulate==0.10.0 \
             tqdm==4.67.1 --quiet

print("Install complete.")

Note: you may need to restart the kernel to use updated packages.
Install complete.


## 2. Imports

In [2]:
import os
import requests
import time

from datetime import datetime, timezone
from neo4j import GraphDatabase
from tabulate import tabulate
from tqdm.auto import tqdm

## 3. Configuration

In [3]:
NEO4J_URI      = os.environ["NEO4J_URI"]
NEO4J_USERNAME = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]
NEO4J_DATABASE = os.environ["NEO4J_DATABASE"]
IQAIR_API_KEY  = os.environ["IQAIR_API_KEY"]

print("Credentials set.")

Credentials set.


In [4]:
# IQAir free tier: 5 requests per minute
API_DELAY_SECONDS = 12

# Cities within this distance will be linked as NEIGHBORS
NEIGHBOR_DISTANCE_KM = 200

# Regions to query
REGIONS = [
    {"country": "France", "states": ["Occitanie", "Nouvelle-Aquitaine"]},
    {"country": "Spain",  "states": ["Aragon", "Catalonia", "Navarre"]},
]

## 4. Connect to Neo4j

In [5]:
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth = (NEO4J_USERNAME, NEO4J_PASSWORD)
)

print(driver.verify_connectivity())  # None is expected
print("Connection created.")

None
Connection created.


In [14]:
# Run this cell ONLY on the first run to clear the database.
# Comment it out on subsequent daily runs to preserve accumulated readings.
# with driver.session(database=NEO4J_DATABASE) as session:
#     session.run("MATCH (n) DETACH DELETE n")
# print("Database cleared.")

Database cleared.


## 5. IQAir Helper Functions

In [6]:
def get_aqi_category(aqi):
    """Map a US AQI value to its standard category label."""
    if aqi <= 50:  return "Good"
    if aqi <= 100: return "Moderate"
    if aqi <= 150: return "Unhealthy for Sensitive Groups"
    if aqi <= 200: return "Unhealthy"
    if aqi <= 300: return "Very Unhealthy"
    return "Hazardous"

def fetch_cities(country, state, api_key):
    """Return a list of city name strings for the given state and country."""
    url    = "http://api.airvisual.com/v2/cities"
    params = {"state": state, "country": country, "key": api_key}
    resp   = requests.get(url, params=params)
    data   = resp.json()
    if data["status"] == "success":
        return [c["city"] for c in data["data"]]
    print(f"  Warning: could not fetch cities for {state}, {country}: {data.get('data')}")
    return []

def fetch_city_data(country, state, city, api_key):
    """Fetch current weather and pollution data for a single city.
    Returns a dict of all fields, or None on failure.
    """
    url    = "http://api.airvisual.com/v2/city"
    params = {"city": city, "state": state, "country": country, "key": api_key}
    resp   = requests.get(url, params=params)
    data   = resp.json()

    if data["status"] != "success":
        print(f"  Warning: no data for {city}: {data.get('data')}")
        return None

    loc       = data["data"]["location"]["coordinates"]
    weather   = data["data"]["current"]["weather"]
    pollution = data["data"]["current"]["pollution"]

    return {
        "country":        country,
        "state":          state,
        "city":           city,
        "lat":            loc[1],
        "lon":            loc[0],
        "timestamp":      datetime.now(timezone.utc).isoformat(),
        "aqi_us":         pollution.get("aqius"),
        "aqi_cn":         pollution.get("aqicn"),
        "aqi_category":   get_aqi_category(pollution.get("aqius", 0)),
        "main_pollutant": pollution.get("mainus"),
        "temperature":    weather.get("tp"),
        "pressure":       weather.get("pr"),
        "humidity":       weather.get("hu"),
        "wind_speed":     weather.get("ws"),
        "wind_direction": weather.get("wd"),
    }

## 6. Neo4j Helper Functions

City coordinates are stored as a native Neo4j `point` type rather than separate `lat` and `lon` float properties. This enables `point.distance()` queries directly in Cypher and supports a point index for efficient spatial lookups.

In [7]:
def create_constraints_and_indexes(session):
    """Create uniqueness constraint and point index on City nodes."""
    session.run("""
        CREATE CONSTRAINT city_unique IF NOT EXISTS
        FOR (c:City) REQUIRE (c.name, c.country) IS UNIQUE
    """)
    session.run("""
        CREATE POINT INDEX city_location IF NOT EXISTS
        FOR (c:City) ON (c.location)
    """)

def upsert_city(session, record):
    """Merge a City node, storing coordinates as a native point type."""
    session.run("""
        MERGE (c:City {name: $city, country: $country})
        SET c.state    = $state,
            c.location = point({latitude: $lat, longitude: $lon})
    """, **record)

def create_reading(session, record):
    """Create a new Reading node and link it to the City."""
    session.run("""
        MATCH (c:City {name: $city, country: $country})
        CREATE (r:Reading {
            timestamp:      $timestamp,
            aqi_us:         $aqi_us,
            aqi_cn:         $aqi_cn,
            aqi_category:   $aqi_category,
            main_pollutant: $main_pollutant,
            temperature:    $temperature,
            pressure:       $pressure,
            humidity:       $humidity,
            wind_speed:     $wind_speed,
            wind_direction: $wind_direction
        })
        CREATE (c)-[:HAS_READING]->(r)
    """, **record)

## 7. Quick Test: City Count by Region

Before running the full pipeline, we fetch only the city lists to check coverage. This uses one API call per state (5 calls total). If any region returns fewer cities than expected, adjust `REGIONS` before proceeding.

In [8]:
test_locations = []

for region in REGIONS:
    country = region["country"]
    print(f"\n{country}:")
    for state in region["states"]:
        cities = fetch_cities(country, state, IQAIR_API_KEY)
        test_locations.extend([(country, state, c) for c in cities])
        print(f"  {state}: {len(cities)} cities")
        time.sleep(API_DELAY_SECONDS)

print(f"\nTotal cities available: {len(test_locations)}")


France:
  Occitanie: 23 cities
  Nouvelle-Aquitaine: 21 cities

Spain:
  Aragon: 4 cities
  Catalonia: 38 cities
  Navarre: 9 cities

Total cities available: 95


## 8. Step 1: Fetch City List from IQAir

In [9]:
locations = []

for region in REGIONS:
    country = region["country"]
    for state in tqdm(region["states"], desc=f"{country}"):
        cities = fetch_cities(country, state, IQAIR_API_KEY)
        for city in cities:
            locations.append((country, state, city))
        print(f"  {state}: {len(cities)} cities")
        time.sleep(API_DELAY_SECONDS)

print(f"\nTotal: {len(locations)} cities")

France:   0%|          | 0/2 [00:00<?, ?it/s]

  Occitanie: 23 cities
  Nouvelle-Aquitaine: 21 cities


Spain:   0%|          | 0/3 [00:00<?, ?it/s]

  Aragon: 4 cities
  Catalonia: 38 cities
  Navarre: 9 cities

Total: 95 cities


## 9. Step 2: Fetch AQI and Weather Data per City

In [10]:
records = []
failed  = []

for country, state, city in tqdm(locations, desc="Fetching AQI data"):
    record = fetch_city_data(country, state, city, IQAIR_API_KEY)
    if record:
        records.append(record)
    else:
        failed.append(f"{city}, {country}")
    time.sleep(API_DELAY_SECONDS)

print(f"Fetched: {len(records)} cities.")
if failed:
    print(f"Failed:  {len(failed)}: {', '.join(failed)}")

Fetching AQI data:   0%|          | 0/95 [00:00<?, ?it/s]

Fetched: 95 cities.


## 10. Step 3: Load Data into Neo4j

We load in two passes:
1. City nodes and Reading nodes
2. `NEIGHBORS` relationships -- computed using `point.distance()` in Cypher, not Python.

In [15]:
with driver.session(database=NEO4J_DATABASE) as session:

    create_constraints_and_indexes(session)
    print("Constraints and indexes created.")

    for record in tqdm(records, desc="Loading cities and readings"):
        upsert_city(session, record)
        create_reading(session, record)

    print(f"Loaded {len(records)} cities and readings.")

Constraints and indexes created.


Loading cities and readings:   0%|          | 0/95 [00:00<?, ?it/s]

Loaded 95 cities and readings.


In [16]:
# Build NEIGHBORS relationships using point.distance() in Cypher.
# No Python math needed -- Neo4j computes the distance natively.
with driver.session(database=NEO4J_DATABASE) as session:
    result = session.run("""
        MATCH (a:City), (b:City)
        WHERE elementId(a) < elementId(b)
          AND point.distance(a.location, b.location) / 1000 <= $max_km
        WITH a, b,
             round(point.distance(a.location, b.location) / 1000, 1) AS distance_km
        MERGE (a)-[r:NEIGHBORS]-(b)
        SET r.distance_km = distance_km
        RETURN count(*) AS neighbors_created
    """, max_km=NEIGHBOR_DISTANCE_KM)
    n = result.single()["neighbors_created"]
    print(f"NEIGHBORS relationships created: {n}")

NEIGHBORS relationships created: 1567


## 11. Step 4: Verify the Data in Neo4j

In [18]:
with driver.session(database=NEO4J_DATABASE) as session:
    result = session.run("MATCH (c:City) RETURN count(c) AS cities")
    print(f"City nodes:              {result.single()['cities']}")
    result = session.run("MATCH (r:Reading) RETURN count(r) AS readings")
    print(f"Reading nodes:           {result.single()['readings']}")
    result = session.run("MATCH ()-[n:NEIGHBORS]-() RETURN count(n) AS neighbors")
    print(f"NEIGHBORS relationships: {result.single()['neighbors']}")

    result = session.run("""
        MATCH (c:City)-[:HAS_READING]->(r:Reading)
        WITH c, r ORDER BY r.timestamp DESC
        WITH c, collect(r)[0] AS latest
        RETURN latest.aqi_us AS aqi_us,
               latest.aqi_category AS category
    """)
    rows = [r.data() for r in result]
    from collections import Counter
    cats = Counter(r["category"] for r in rows)
    print(f"\nAQI range:   {min(r['aqi_us'] for r in rows)} - {max(r['aqi_us'] for r in rows)}")
    print(f"Average AQI: {sum(r['aqi_us'] for r in rows) / len(rows):.1f}")
    print("\nBy category:")
    for cat, count in sorted(cats.items(), key=lambda x: x[1], reverse=True):
        print(f"  {cat}: {count}")

City nodes:              95
Reading nodes:           95
NEIGHBORS relationships: 3134

AQI range:   1 - 74
Average AQI: 36.9

By category:
  Good: 73
  Moderate: 22


## 12. Teardown

In [19]:
driver.close()
print("Connection closed.")

Connection closed.
